In [3]:
import torch
# 叶子节点
X1 = torch.tensor(2.0, requires_grad=True)
X2 = torch.tensor(3.0, requires_grad=True)

a = X1 * X2                  # a = X1 * X2
y1 = torch.log(a)            # y1 = log(a)
y2 = torch.sin(X2)           # y2 = sin(X2)
w = y1 * y2                  # w = y1 * y2
z = w                        # z = w

a.retain_grad()
y1.retain_grad()
y2.retain_grad()
w.retain_grad()
z.retain_grad()

print(y1.data, y1.grad, y1.grad_fn, y1.retain_grad())

z.backward()

print(y1.data, y1.grad, y1.grad_fn, y1.retain_grad())


tensor(1.7918) None <LogBackward0 object at 0x76abb499fb50> None
tensor(1.7918) tensor(0.1411) <LogBackward0 object at 0x76abb47e5a20> None


实现各类分布式训练，测试模型主要是两类（首先本地vGPU测试单卡调试没问题而后去多卡进行测试）：1、视觉模型直接使用Resnet50然后在CIFAR10数据集上进行训练；2、使用Qwen0.5B模型进行文本分类训练

**代码要求**：1、尽可能的封装，方便直接用来进行复现代码

**文档要求**：1、在每一个分布式训练中都需要写好参数、参考地址、安装配置等尽可能详细就好；2、由于原理相通，所以**尽量多的去介绍调用框架的参数，对于具体原理不需要过多进行介绍**

# 基于原生 Pytorch 进行分布式训练

## DDP
> https://docs.pytorch.org/docs/2.11/notes/ddp.html


- [ ] 了解DDP底层：模型分配、如何执行 all-reduce操作
- [ ] 总结学习：https://docs.pytorch.org/docs/2.11/distributed.html#torch.distributed.all_reduce 其中所有带 distributed 的页面

`DDP`训练过程中原理：为不同设备都去复制一遍模型而后在不同的设备上都去输入一批数据在分别计算得到梯度之后进行 `all-reduce` 操作处理即可得到更新后模型在进行下一步优化。因此对于 `DDP`就需要关注几个点：1、模型分配；2、不同设备之间通信；3、数据处理。
### DDP 训练简易模板
测试代码：[torchDDP_training](/learning_distribute_training/torchDDP_training.py)对于代码执行直接 `torchrun --nproc_per_node=4 train_script.py`，上述命令会启动 4 个 Python 进程，并自动为它们设置不同的环境变量：

进程 0: 设置 RANK=0, LOCAL_RANK=0, WORLD_SIZE=4

进程 1: 设置 RANK=1, LOCAL_RANK=1, WORLD_SIZE=4

进程 2: 设置 RANK=2, LOCAL_RANK=2, WORLD_SIZE=4

进程 3: 设置 RANK=3, LOCAL_RANK=3, WORLD_SIZE=4

那么在代码里直接使用 `os.environ.get("RANK")` 能直接拿到每一个显卡的“值”。**值得注意的是**：rank是所有的设备的序号，而 local_rank 是本机的序号，比如说有2台机器每一台都是4卡那么rank序号就是从0->7而 local_rank 则是从0->3而后另外一台机器也是 0->3

对于上述代码简单总结如下：**首先**数据处理，因为DDP中需要对不同设备进行数据处理，因此对于不同的数据集需要设定不同的 `sampler` 具体使用直接使用 `DistributedSampler` 而后构建得到不同的 dataloader，**而后**就是对于模型处理可以直接使用 `self.model = DDP(..., device_ids=..., output_device=...)` 将模型进行处理，不过 **值得注意的是**，在通过DDP处理后模型在对模型保存需要直接通过 `self.model.module` 拿出模型，**最后**模型训练（对应内部的 `training_step`），模型训练过程中考虑：1、混合精度训练；2、梯度累计训练；2、grad checkpoint处理方式等，对于混合精度训练：**计算得到loss之后使用混合训练去处理loss以及梯度**（对loss以及梯度精度转换），对于 **梯度累计过程**只需要对根据累计步数确定是不是进行 optimizer 更新其他没有处理。

###  DDP训练 底层原理

## FSDP

### FSDP 训练简易模板
对于FSDP简单理解为直接基于 torch 的DeepSpeed实现，因此在使用FSDP过程中对于torch的支持也很好，比如说在使用FSDP进行训练过程中一般而言简单的模板为：
```python
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    FullOptimStateDictConfig,
    FullStateDictConfig,
    ShardingStrategy,
    StateDictType,
)
# 类似DDP 中训练一样 首先去对模型进行处理
self.model = FSDP(
                self.model,
                device_id=self.local_rank if self.device.type == "cuda" else None,
                sharding_strategy=sharding_strategy,
                use_orig_params=bool(getattr(self.config, "fsdp_use_orig_params", True)),
                limit_all_gathers=bool(getattr(self.config, "fsdp_limit_all_gathers", True)),
                sync_module_states=bool(getattr(self.config, "fsdp_sync_module_states", False)),
            )
# 后续训练过程没有差异
```
**[FSDP参数解释](https://docs.pytorch.org/docs/2.12/fsdp.html)**：

`sharding_strategy`：[分片策略](https://docs.pytorch.org/docs/2.12/fsdp.html#torch.distributed.fsdp.ShardingStrategy)，支持有：1、`FULL_SHARD`；2、`SHARD_GRAD_OP` — 只分片 梯度 + 优化器状态，参数不分片但每个 rank 都有一份完整拷贝；3、`NO_SHARD` — 不分片，等同于 DDP；4、`HYBRID_SHARD` — 混合分片（节点内分片，节点间复制）

`use_orig_params`：是否使用原始参数；`limit_all_gathers`：是否限制并发的 All-Gather 操作数量，True：只在forward以及backwards过程中启动all-gather，False：同时处理多个all-gather；`sync_module_states`：是否在 FSDP 初始化后，将 rank 0 的模型参数广播同步到所有其他 rank

- [ ] torch.compile 支持
- [ ] 更加多测试，如 dit 模型测试
- [ ] 保存训练参数配置，写到 tensorboard 文件夹中

# 基于 accelerate 进行分布式训练
> https://huggingface.co/docs/accelerate/index

最简单使用可以直接使用 `from transformers import trainer` 内部逻辑直接基于 accelerate 进行处理，支持 `deepepeed`、`fsdp`等训练方式
## 简单使用
**安装过程**直接通过：`pip install accelerate` 即可，使用过程直接通过如下命令：

```python
from accelerate import Accelerator
accelerator = Accelerator() # 初始化：指定日志记录（tensorboard等）、混合精度训练等

device = accelerator.device
model, optimizer, training_dataloader, scheduler = accelerator.prepare(
    model, optimizer, training_dataloader, scheduler
) # 通过 prepare 函数将模型、优化器、数据加载器、学习率调度器进行加速

for batch in training_dataloader:
    optimizer.zero_grad()
    inputs, targets = batch
    outputs = model(inputs)
    loss = loss_function(outputs, targets)
    # 区别普通的训练过程 accelerator 直接通过 backward 函数进行反向传播
    accelerator.backward(loss)
    optimizer.step()
    scheduler.step()
```

定义完代码之后启动脚本直接通过（对于启动支持的命令 `accelerate launch -h` 即可查看所有支持命令）：`accelerate launch main.py {--arg1} {--arg2}` **值得注意的是**，启动脚本中可以直接通过 `accelerate config` 而后会在终端中进行配置最后得到一个 `yaml` 的 config 文件，然后启动过程中直接：`accelerate launch --config_file {path/to/config/my_config_file.yaml} main.py {--arg1} {--arg2}` 即可，亦或者不想去使用 `yaml` 文件配置，可以直接在代码中指定好配置参数（比如直接使用 dataclass 去封装好训练参数），而后直接 `CUDA_VISIBLE_DEVICES=0,5,6,7 accelerate launch train_sft.py`（通过`CUDA_VISIBLE_DEVICES` 指定使用哪些 GPU进行训练）

## 初始化参数配置
根据[官方](https://huggingface.co/docs/accelerate/package_reference/accelerator)中参数配置主要使用参数如下（因为对于多种分布式训练如：DeepSpeed、FSDP等都在初始化配置中进行配置即可，因此下面代码中不去介绍这些参数，只介绍最常用的参数，在**后续具体使用过程中再去着重介绍**）：
```python
from accelerate import Accelerator

accelerator = Accelerator(
    mixed_precision= config.mixed_precision, # 混合精度训练: str=fp16/bf16/fp8/no
    gradient_accumulation_steps= config.gradient_accumulation_steps, # 梯度累积：int
    log_with= config.report_to, # 日志记录 list/str = ['wandb','tensorboard']等
    project_config= accelerator_project_config, # 项目配置 所有的文件保存路径
    kwargs_handlers= kwargs_handlers, # 配置参数处理
    )
```

## 分布式训练配置
### DDP
在`accelerate`中默认直接使用DDP训练因此不需要太多参数配置
### DeepSpeed

### FSDP2


# 基于 DeepSpeed 进行分布式训练

# 基于 Megatron-LM 进行分布式训练
> [https://github.com/nvidia/megatron-lm](https://github.com/nvidia/megatron-lm)

# 使用 Ray 分布式框架
https://docs.pytorch.org/tutorials/beginner/distributed_training_with_ray_tutorial.html